In [1]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np
from sklearn.metrics import precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
import re
import os
from itertools import combinations
from statsmodels.stats.multitest import multipletests


The purpose of this notebook is to create

- Figure 2a: a bar plot comparing the performance (precision and recall) of the models on the cogntive status tasks (NC/MCI/DE), with stats using bootstrap.
- A table containing the same data, plus the F1 score. The table is in LaTeX format, ready to be pasted into the manuscript.

In [2]:
nifd_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/NIFD/test_cog')
nacc_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/NACC/test_cog')
adni_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/ADNI/test_cog')
ppmi_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/PPMI/test_cog')
brainlat_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/brainlat/test_cog')

The following functions load the data from the paths specified above, pick only relevant columns, and concatenate everything. 

Later we compute class-wise precision and recall, and estimate error bars on them using bootstrap. The bootstrap samples are kept, so we can compute anything else we want after resampling, ensuring that the calculations use the correct samples.

In [3]:
def option_string_to_dict(options):
    # The option string is randomized (e.g. NC is not always option A). We need to break down the 
    # options and look at the text (e.g. MCI), not just the letter that identifies them in a particular question
    pattern = r"([A-Z])\. ([^\n]+)"
    matches = re.findall(pattern, options)
    return {key: value for key, value in matches}

def load_answers(dir_path, dataset_name):
    # load all parquet files from the directory, stack them into a pandas datafame
    # this only reads the participant ID, ground trush answer and the prediction. 
    # Reading only those columns is significantly faster (about 100x) than loading the whole dataframe.
    # Loading everything is very slow because there are extremely long strings (model outputs) in some columns

    fpaths = list(dir_path.rglob('*.parquet'))

    dfs = []

    cols_to_read = ['ID','ground_truth','prediction','ground_truth_text','options']

    for fpath in tqdm(fpaths):

        model = fpath.parent.name.split('-',3)[-1] 
        benchmark = fpath.parent.parent.name.split('_',1)[-1].upper()

        df = pd.read_parquet(fpath,columns=cols_to_read)
    
        df = df.assign(model=model, benchmark=benchmark)

        df['correct'] = (df['ground_truth'] == df['prediction']).astype(int)

        df['prediction_text'] = df.apply(lambda row: option_string_to_dict(row['options']).get(row['prediction'],'invalid'),axis=1)

        dfs.append(df)

    df = pd.concat(dfs)
    df['dataset'] = dataset_name

    # make these columns Categorical
    group_cols = ["dataset", "benchmark", "model", "ground_truth_text", 'prediction_text']
    for col in group_cols:
        df[col] = pd.Categorical(df[col])

    # keep only the models we actually care about
    df = df[df['model'].isin(['Qwen2.5-3B-Instruct','Qwen2.5-7B-Instruct','NACC-3B-OS-SCE'])]

    return df


The `dataset_name` parameter will be used in the results dataset to identify which dataset the data came from

In [4]:
nacc_res = load_answers(nacc_path,dataset_name='NACC')

100%|██████████| 8/8 [00:19<00:00,  2.48s/it]


In [5]:
brainlat_res = load_answers(brainlat_path,dataset_name='BrainLat')

100%|██████████| 8/8 [00:01<00:00,  7.71it/s]


In [6]:
ppmi_res = load_answers(ppmi_path,dataset_name='PPMI')

100%|██████████| 8/8 [00:02<00:00,  3.12it/s]


In [7]:
nifd_res = load_answers(nifd_path,dataset_name='NIFD')

100%|██████████| 8/8 [00:00<00:00, 14.81it/s]


In [8]:
adni_res = load_answers(adni_path,dataset_name='ADNI')

100%|██████████| 8/8 [00:00<00:00,  9.91it/s]


In [9]:
# concatenate everything in a tall format dataframe

results_df = pd.concat(
    [
        adni_res,
        brainlat_res,
        nifd_res,
        nacc_res,
        ppmi_res
    ]
)

In [10]:
# example of what the dataframe with the task results look like
results_df.sample(3)

,ID,ground_truth,prediction,ground_truth_text,options,model,benchmark,correct,prediction_text,dataset
106213,NACC515275,A,A,Normal Cognition (NC),A. Normal Cognition (NC)\nB. Dementia (DE)\nC....,Qwen2.5-3B-Instruct,COG,1,Normal Cognition (NC),NACC
3252,6800,A,A,Mild Cognitive Impairment (MCI),A. Mild Cognitive Impairment (MCI)\nB. Normal ...,Qwen2.5-7B-Instruct,COG,1,Mild Cognitive Impairment (MCI),ADNI
150748,NACC498167,A,A,Normal Cognition (NC),A. Normal Cognition (NC)\nB. Mild Cognitive Im...,NACC-3B-OS-SCE,COG,1,Normal Cognition (NC),NACC


In [11]:
def _process_group_with_bootstrap_samples(id, group, n_boot, seed):
    """
    Modified helper function that returns raw bootstrap samples in addition to CIs.
    This allows us to perform statistical tests on the bootstrap distributions.
    """
    dataset, model = id
    n_samples = len(group)
    
    if n_samples == 0:
        return [], {}

    unique_labels = group['ground_truth_text'].unique()
    
    # --- Reproducible RNG per worker ---
    rng = np.random.default_rng(seed)
    
    # Pre-generate all bootstrap indices at once
    bootstrap_indices = rng.integers(0, n_samples, size=(n_boot, n_samples))
    
    # Pre-allocate dicts to store results per class
    boot_results = {}
    for label in unique_labels:
        boot_results[('precision', label)] = []
        boot_results[('recall', label)] = []
    
    # Convert to NumPy arrays ONCE per group
    y_true = group["ground_truth_text"].to_numpy()
    y_pred = group["prediction_text"].to_numpy()
    
    for indices in bootstrap_indices:
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]

        # Get per-class metrics
        p, r, f, s = precision_recall_fscore_support(
            y_true_boot,
            y_pred_boot,
            average=None,
            labels=unique_labels,
            zero_division=0,
        )
        
        # Store per-class results
        for i, label in enumerate(unique_labels):
            boot_results[('precision', label)].append(p[i])
            boot_results[('recall', label)].append(r[i])

    # --- Calculate quantiles ---
    res_list = []
    bootstrap_samples = {}  # Store raw samples for statistical tests
    
    for label in unique_labels:
        # Process precision for this class
        precision_values = np.array(boot_results[('precision', label)])
        
        low_idx = int(0.025 * n_boot)
        med_idx = int(0.5 * n_boot)
        high_idx = int(0.975 * n_boot)
        
        partitioned = np.partition(precision_values, [low_idx, med_idx, high_idx])
        
        res_list.append({
            "dataset": dataset,
            "model": model,
            "class": label,
            "metric": "precision",
            "median": partitioned[med_idx],
            "low": partitioned[low_idx],
            "high": partitioned[high_idx],
        })
        
        # Store bootstrap samples
        bootstrap_samples[(dataset, model, label, "precision")] = precision_values
        
        # Process recall for this class
        recall_values = np.array(boot_results[('recall', label)])
        partitioned = np.partition(recall_values, [low_idx, med_idx, high_idx])
        
        res_list.append({
            "dataset": dataset,
            "model": model,
            "class": label,
            "metric": "recall",
            "median": partitioned[med_idx],
            "low": partitioned[low_idx],
            "high": partitioned[high_idx],
        })
        
        # Store bootstrap samples
        bootstrap_samples[(dataset, model, label, "recall")] = recall_values
        
    return res_list, bootstrap_samples


def optimized_bootstrap_parallel_with_samples(df, n_boot, seed=None, n_jobs=-1):
    """
    Compute bootstrap confidence intervals and return raw bootstrap samples.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with columns: dataset, model, ground_truth_text, prediction_text
    n_boot : int
        Number of bootstrap samples
    seed : int, optional
        Random seed for reproducible results
    n_jobs : int, optional
        Number of CPU cores to use. -1 means all cores.
    
    Returns
    -------
    results_df : pd.DataFrame
        Results with columns: dataset, model, class, metric, median, low, high
    bootstrap_samples : dict
        Dictionary mapping (dataset, model, class, metric) -> array of bootstrap samples
    """
    
    # --- Setup ---
    main_rng = np.random.default_rng(seed)
    
    df_copy = df.copy()
    group_cols = ["dataset", "model"]
            
    # Get all groups
    groups = list(df_copy.groupby(group_cols, observed=True))
    
    # Generate a unique seed for each group
    n_groups = len(groups)
    group_seeds = main_rng.integers(0, 2**32, size=n_groups)
    
    # --- Parallel Execution ---
    results_with_samples = Parallel(n_jobs=n_jobs, verbose=0)(
        delayed(_process_group_with_bootstrap_samples)(
            id, 
            group, 
            n_boot, 
            group_seeds[i]
        )
        for i, (id, group) in enumerate(groups)
    )
    
    # --- Collect Results ---
    final_results = []
    all_bootstrap_samples = {}
    
    for res_list, boot_samples in results_with_samples:
        final_results.extend(res_list)
        all_bootstrap_samples.update(boot_samples)
    
    return pd.DataFrame(final_results), all_bootstrap_samples


def permutation_test_bootstrap_medians(samples1, samples2, n_permutations=10000, seed=None):
    """
    Perform a permutation test comparing medians of two bootstrap distributions.
    
    Parameters
    ----------
    samples1 : np.ndarray
        Bootstrap samples from first model
    samples2 : np.ndarray
        Bootstrap samples from second model
    n_permutations : int
        Number of permutations to perform
    seed : int, optional
        Random seed for reproducibility
    
    Returns
    -------
    p_value : float
        Two-sided p-value
    """
    rng = np.random.default_rng(seed)
    
    # Observed difference in medians
    observed_diff = np.median(samples1) - np.median(samples2)
    
    # Combine samples
    combined = np.concatenate([samples1, samples2])
    n1 = len(samples1)
    n_total = len(combined)
    
    # Permutation test
    null_diffs = np.zeros(n_permutations)
    
    for i in range(n_permutations):
        # Shuffle combined samples
        perm_idx = rng.permutation(n_total)
        perm1 = combined[perm_idx[:n1]]
        perm2 = combined[perm_idx[n1:]]
        
        # Compute difference in medians under null
        null_diffs[i] = np.median(perm1) - np.median(perm2)
    
    # Two-sided p-value
    p_value = np.mean(np.abs(null_diffs) >= np.abs(observed_diff))
    
    return p_value


def compute_pairwise_comparisons(bootstrap_samples, n_permutations=10000, seed=None, n_jobs=-1):
    """
    Compute pairwise permutation tests between all models for each dataset/class/metric combination.
    
    Parameters
    ----------
    bootstrap_samples : dict
        Dictionary from optimized_bootstrap_parallel_with_samples
        Maps (dataset, model, class, metric) -> array of bootstrap samples
    n_permutations : int
        Number of permutations for each test
    seed : int, optional
        Random seed for reproducibility
    n_jobs : int, optional
        Number of CPU cores to use. -1 means all cores.
    
    Returns
    -------
    pd.DataFrame
        Results with columns: dataset, class, metric, model1, model2, p_value
    """
    
    # Extract unique combinations of dataset, class, metric
    keys = list(bootstrap_samples.keys())
    unique_combos = set((dataset, cls, metric) for dataset, model, cls, metric in keys)
    
    # For each combination, get all models
    comparison_tasks = []
    
    for dataset, cls, metric in unique_combos:
        # Find all models for this combination
        models = [model for d, model, c, m in keys 
                  if d == dataset and c == cls and m == metric]
        models = sorted(set(models))
        
        # Generate all pairwise comparisons
        for model1, model2 in combinations(models, 2):
            key1 = (dataset, model1, cls, metric)
            key2 = (dataset, model2, cls, metric)
            
            if key1 in bootstrap_samples and key2 in bootstrap_samples:
                comparison_tasks.append({
                    'dataset': dataset,
                    'class': cls,
                    'metric': metric,
                    'model1': model1,
                    'model2': model2,
                    'samples1': bootstrap_samples[key1],
                    'samples2': bootstrap_samples[key2]
                })
    
    # Set up RNG for reproducibility
    main_rng = np.random.default_rng(seed)
    task_seeds = main_rng.integers(0, 2**32, size=len(comparison_tasks))
    
    # Parallel execution of permutation tests
    def run_single_test(task, task_seed):
        p_value = permutation_test_bootstrap_medians(
            task['samples1'],
            task['samples2'],
            n_permutations=n_permutations,
            seed=task_seed
        )
        return {
            'dataset': task['dataset'],
            'class': task['class'],
            'metric': task['metric'],
            'model1': task['model1'],
            'model2': task['model2'],
            'p_value': p_value,
            'median1': np.median(task['samples1']),
            'median2': np.median(task['samples2']),
            'median_diff': np.median(task['samples1']) - np.median(task['samples2'])
        }
    
    results = Parallel(n_jobs=n_jobs, verbose=0)(
        delayed(run_single_test)(task, task_seeds[i])
        for i, task in enumerate(comparison_tasks)
    )

    df = pd.DataFrame(results)

    # Benjamini–Hochberg correction per (dataset, class, metric)
    df['p_value_bh'] = np.nan

    for (_, _, _), idx in df.groupby(['dataset', 'class', 'metric']).groups.items():
        pvals = df.loc[idx, 'p_value'].values
        _, pvals_bh, _, _ = multipletests(pvals, method='fdr_bh',alpha=0.05)
        df.loc[idx, 'p_value_bh'] = pvals_bh

    
    return df

We are going to lump together all external validation datasets, and treat separately only NACC (Internal validation) vs Every other cohort (External validation)

In [12]:
results_df['dataset_raw'] = results_df['dataset']

results_df['dataset'] = results_df['dataset'].replace(
    {
        'ADNI':'Other',
        'BrainLat':'Other',
        'NIFD':'Other',
        'PPMI':'Other',
    }
)

In [13]:
results_df.sample(5)

,ID,ground_truth,prediction,ground_truth_text,options,model,benchmark,correct,prediction_text,dataset,dataset_raw
93414,NACC179563,B,B,Dementia (DE),A. Mild Cognitive Impairment (MCI)\nB. Dementi...,Qwen2.5-7B-Instruct,COG,1,Dementia (DE),NACC,NACC
48732,NACC707397,A,A,Dementia (DE),A. Dementia (DE)\nB. Mild Cognitive Impairment...,Qwen2.5-7B-Instruct,COG,1,Dementia (DE),NACC,NACC
68546,NACC172611,C,B,Mild Cognitive Impairment (MCI),A. Dementia (DE)\nB. Normal Cognition (NC)\nC....,Qwen2.5-3B-Instruct,COG,0,Normal Cognition (NC),NACC,NACC
86004,NACC452516,B,B,Dementia (DE),A. Mild Cognitive Impairment (MCI)\nB. Dementi...,Qwen2.5-3B-Instruct,COG,1,Dementia (DE),NACC,NACC
65470,NACC444600,B,C,Normal Cognition (NC),A. Dementia (DE)\nB. Normal Cognition (NC)\nC....,NACC-3B-OS-SCE,COG,0,Mild Cognitive Impairment (MCI),NACC,NACC


The following function both computes bootstrap stats on precision and recall (used in the figure later), and returns the raw bootstrap samples. 

The raw samples are used to compute the F1 score, and are available if we later want to compute something else. The F1 score is only presented in the table, not in the figure.

In [14]:
# Compute bootstrap CIs with samples
all_metrics, bootstrap_samples = optimized_bootstrap_parallel_with_samples(
    results_df, n_boot=10, seed=42, n_jobs=-1
)

In [15]:
def add_f1_scores(metrics_dict):
    """
    Takes a dictionary with precision and recall bootstrap samples and adds F1 scores.
    
    Parameters:
    -----------
    metrics_dict : dict
        Dictionary with keys as tuples ending in 'precision' or 'recall',
        and values as numpy arrays of bootstrap samples.
    
    Returns:
    --------
    dict
        Original dictionary with added F1 score entries.
    """
    # Create a copy to avoid modifying the original
    result_dict = metrics_dict.copy()
    
    # Find all precision keys and compute corresponding F1 scores
    for key in metrics_dict.keys():
        if key[-1] == 'precision':
            # Create corresponding recall and f1 keys
            prefix = key[:-1]
            recall_key = prefix + ('recall',)
            f1_key = prefix + ('f1',)
            
            precision = metrics_dict[key]
            recall = metrics_dict[recall_key]
            
            # Compute F1 score: 2 * (precision * recall) / (precision + recall)
            with np.errstate(divide='ignore', invalid='ignore'):
                f1 = 2 * (precision * recall) / (precision + recall)
                # Set F1 to 0 where both precision and recall are 0
                f1 = np.where(np.isnan(f1), 0, f1)
            
            result_dict[f1_key] = f1
    
    return result_dict

In [16]:
def create_summary_table(metrics_dict, model_map=None, class_map=None, 
                         label_columns=None, sort_by=None,
                         class_order=None, model_order=None):
    """
    Creates a summary table with median and 95% CI for each metric.
    
    Parameters:
    -----------
    metrics_dict : dict
        Dictionary with keys as tuples and values as numpy arrays of bootstrap samples.
    model_map : dict, optional
        Mapping from original model names to abbreviations
    class_map : dict, optional
        Mapping from original class names to abbreviations
    label_columns : list, optional
        Order of label columns to display (e.g., ['Dataset', 'Model', 'Class'])
        Default: ['Dataset', 'Model', 'Class']
    sort_by : list, optional
        Columns to sort by. Default: same as label_columns
    class_order : list, optional
        Order for Class categorical (e.g., ['NC', 'MCI', 'DE'])
    model_order : list, optional
        Order for Model categorical (e.g., ['Q3B', 'LUNAR', 'Q7B'])
    
    Returns:
    --------
    pd.DataFrame
        Pivoted table with metrics as columns showing median and 95% CI.
    """
    if label_columns is None:
        label_columns = ['Dataset', 'Model', 'Class']
    if sort_by is None:
        sort_by = label_columns
    
    rows = []
    
    for key, values in metrics_dict.items():
        # Extract components from the key
        *prefix, metric_name = key
        
        # Compute median and 95% CI (2.5th and 97.5th percentiles)
        median = np.median(values)
        ci_lower = np.percentile(values, 2.5)
        ci_upper = np.percentile(values, 97.5)
        
        # Extract and map components
        dataset = prefix[0] if len(prefix) > 0 else ''
        model = prefix[1] if len(prefix) > 1 else ''
        class_name = prefix[2] if len(prefix) > 2 else ''
        
        # Apply mappings if provided
        if model_map and model in model_map:
            model = model_map[model]
        if class_map and class_name in class_map:
            class_name = class_map[class_name]
        
        # Create row
        row = {
            'Dataset': dataset,
            'Model': model,
            'Class': class_name,
            'Metric': metric_name,
            'Value': f"{median:.3f} [{ci_lower:.3f}, {ci_upper:.3f}]"
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    
    # Convert to categorical with specified order
    if class_order is not None and 'Class' in df.columns:
        df['Class'] = pd.Categorical(df['Class'], categories=class_order, ordered=True)
    
    if model_order is not None and 'Model' in df.columns:
        df['Model'] = pd.Categorical(df['Model'], categories=model_order, ordered=True)
    
    # Pivot to have metrics as columns
    df_pivot = df.pivot_table(
        index=label_columns,
        columns='Metric',
        values='Value',
        aggfunc='first',
        observed=False
    ).reset_index()
    
    # Remove the column index name
    df_pivot.columns.name = None
    
    # Reorder columns: label columns first, then metrics
    metric_order = ['precision', 'recall', 'f1']
    available_metrics = [m for m in metric_order if m in df_pivot.columns]
    df_pivot = df_pivot[label_columns + available_metrics]
    
    # Sort by specified columns
    df_pivot = df_pivot.sort_values(sort_by).reset_index(drop=True)
    
    return df_pivot

In [17]:
def format_for_latex(df):
    """
    Formats the pivoted dataframe for LaTeX export.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Pivoted summary table from create_summary_table
    
    Returns:
    --------
    str
        LaTeX table string
    """
    latex_str = df.to_latex(index=False, escape=False)
    return latex_str

In [18]:
# Define mappings
model_map = {
    "Qwen2.5-3B-Instruct": "Q3B",
    "NACC-3B-OS-SCE": "LUNAR",
    "Qwen2.5-7B-Instruct": "Q7B",
}

class_map = {
    'Mild Cognitive Impairment (MCI)': 'MCI',
    'Dementia (DE)': 'DE',
    'Normal Cognition (NC)': 'NC'
}

# Add F1 scores
metrics_with_f1 = add_f1_scores(bootstrap_samples)

# Create table with categorical ordering
summary_df = create_summary_table(
    metrics_with_f1, 
    model_map=model_map, 
    class_map=class_map,
    label_columns=['Dataset', 'Class', 'Model'],
    sort_by=['Dataset', 'Class', 'Model'],
    class_order=['NC', 'MCI', 'DE'],
    model_order=['Q3B', 'LUNAR', 'Q7B']
).replace({
'NACC':'NACC (Internal testing)',
'Other':'All other cohorts (External Testing)'
})

# Display and export
display(summary_df)

latex_table = format_for_latex(summary_df)
with open('results_table.tex', 'w') as f:
    f.write(latex_table)

,Dataset,Class,Model,precision,recall,f1
0,NACC (Internal testing),NC,Q3B,"0.895 [0.893, 0.898]","0.684 [0.682, 0.686]","0.776 [0.774, 0.777]"
1,NACC (Internal testing),NC,LUNAR,"0.927 [0.926, 0.928]","0.817 [0.814, 0.819]","0.868 [0.866, 0.870]"
2,NACC (Internal testing),NC,Q7B,"0.892 [0.891, 0.894]","0.809 [0.807, 0.810]","0.848 [0.847, 0.850]"
3,NACC (Internal testing),MCI,Q3B,"0.277 [0.274, 0.279]","0.694 [0.691, 0.697]","0.396 [0.392, 0.398]"
4,NACC (Internal testing),MCI,LUNAR,"0.428 [0.427, 0.435]","0.693 [0.689, 0.698]","0.530 [0.528, 0.535]"
5,NACC (Internal testing),MCI,Q7B,"0.331 [0.329, 0.334]","0.709 [0.706, 0.715]","0.451 [0.449, 0.455]"
6,NACC (Internal testing),DE,Q3B,"0.892 [0.890, 0.894]","0.594 [0.592, 0.598]","0.713 [0.711, 0.717]"
7,NACC (Internal testing),DE,LUNAR,"0.932 [0.931, 0.933]","0.841 [0.838, 0.843]","0.884 [0.882, 0.886]"
8,NACC (Internal testing),DE,Q7B,"0.965 [0.964, 0.967]","0.639 [0.637, 0.640]","0.769 [0.767, 0.770]"
9,All other cohorts (External Testing),NC,Q3B,"0.848 [0.845, 0.854]","0.476 [0.471, 0.481]","0.609 [0.606, 0.615]"


In [19]:
# Step 2: Compute pairwise comparisons
pairwise_pvalues = compute_pairwise_comparisons(
    bootstrap_samples, n_permutations=10000, seed=42, n_jobs=-1
)

In [20]:
pairwise_pvalues.sort_values(['dataset','metric']).head(10)

,dataset,class,metric,model1,model2,p_value,median1,median2,median_diff,p_value_bh
15,NACC,Dementia (DE),precision,NACC-3B-OS-SCE,Qwen2.5-3B-Instruct,0.0006,0.931705,0.892227,0.039478,0.0009
16,NACC,Dementia (DE),precision,NACC-3B-OS-SCE,Qwen2.5-7B-Instruct,0.0007,0.931705,0.965492,-0.033787,0.0009
17,NACC,Dementia (DE),precision,Qwen2.5-3B-Instruct,Qwen2.5-7B-Instruct,0.0009,0.892227,0.965492,-0.073265,0.0009
21,NACC,Mild Cognitive Impairment (MCI),precision,NACC-3B-OS-SCE,Qwen2.5-3B-Instruct,0.0010,0.427946,0.277005,0.150940,0.0010
22,NACC,Mild Cognitive Impairment (MCI),precision,NACC-3B-OS-SCE,Qwen2.5-7B-Instruct,0.0004,0.427946,0.330982,0.096963,0.0010
23,NACC,Mild Cognitive Impairment (MCI),precision,Qwen2.5-3B-Instruct,Qwen2.5-7B-Instruct,0.0007,0.277005,0.330982,-0.053977,0.0010
33,NACC,Normal Cognition (NC),precision,NACC-3B-OS-SCE,Qwen2.5-3B-Instruct,0.0006,0.926772,0.895397,0.031375,0.0009
34,NACC,Normal Cognition (NC),precision,NACC-3B-OS-SCE,Qwen2.5-7B-Instruct,0.0011,0.926772,0.891789,0.034984,0.0011
35,NACC,Normal Cognition (NC),precision,Qwen2.5-3B-Instruct,Qwen2.5-7B-Instruct,0.0003,0.895397,0.891789,0.003609,0.0009
0,NACC,Normal Cognition (NC),recall,NACC-3B-OS-SCE,Qwen2.5-3B-Instruct,0.0004,0.816910,0.683804,0.133107,0.0012


The following cell contains the code to create the plot 

In [21]:
sns.set_style("whitegrid")


def get_significance_marker(p_value):
    """Convert p-value to significance marker."""
    if p_value < 0.001:
        return "***"
    elif p_value < 0.01:
        return "**"
    elif p_value < 0.05:
        return "*"
    else:
        return "ns"  # Not significant


# def format_pvalue(p_value):
#     """Format p-value for display (alternative version)."""
#     if p_value < 0.001:
#         return "p<0.001"
#     else:
#         return f"p={p_value:.3f}"


def plot_classwise_with_pvalues(all_metrics, pairwise_pvalues, 
                                  model_map, class_map,
                                  output_dir="../figure2_classwise",
                                  show_all_comparisons=False,
                                  p_threshold=0.05):
    """
    Plot classwise precision and recall with p-value annotations.
    
    Parameters
    ----------
    all_metrics : pd.DataFrame
        Results with columns: dataset, model, class, metric, median, low, high
    pairwise_pvalues : pd.DataFrame
        P-values with columns: dataset, class, metric, model1, model2, p_value
    model_map : dict
        Mapping from full model names to abbreviations
    class_map : dict
        Mapping from full class names to abbreviations
    output_dir : str
        Directory to save the figure
    show_all_comparisons : bool
        If True, show all pairwise comparisons. If False, only show comparisons
        involving the first model (typically baseline)
    p_threshold : float
        Only show annotations for p-values below this threshold
    """
    
    # Apply mappings
    all_metrics = all_metrics.copy()
    all_metrics["model_abbrev"] = all_metrics["model"].map(model_map)
    all_metrics["class_abbrev"] = all_metrics["class"].map(class_map).fillna(all_metrics["class"])
    
    # Apply mappings to pairwise_pvalues
    pvalues = pairwise_pvalues.copy()

    # Map abbreviated names to full names for matching
    pvalues["model1_abbrev"] = pvalues["model1"].apply(
        lambda x: model_map.get(x, x)
    )
    pvalues["model2_abbrev"] = pvalues["model2"].apply(
        lambda x: model_map.get(x, x)
    )
    pvalues["class_abbrev"] = pvalues["class"].map(class_map).fillna(pvalues["class"])
    
    # Filter for precision and recall only
    df = all_metrics[all_metrics['metric'].isin(['precision', 'recall'])].copy()
    
    # Get unique values
    datasets = sorted(df["dataset"].unique())
    models = ["Q3B", "LUNAR", "Q7B"]
    
    # Define colors for models
    palette = dict(zip(models, sns.color_palette("colorblind", n_colors=len(models))))
    
    # Create figure: rows = metrics, columns = datasets
    metrics = ['precision', 'recall']
    metric_names = {'precision': 'Precision', 'recall': 'Recall'}
    n_datasets = len(datasets)
    
    fig, axes = plt.subplots(2, n_datasets, figsize=(4 * n_datasets, 6))
    
    # Ensure axes is 2D even with one dataset
    if n_datasets == 1:
        axes = axes.reshape(-1, 1)
    
    # Fixed orders
    class_order = ["NC", "MCI", "IMP", "DE"]
    # dataset_order = ["NACC", "ADNI", "BrainLat", "NIFD", "PPMI"]
    dataset_order = ["NACC", "Other"]
    
    # Override datasets list
    datasets = [d for d in dataset_order if d in df["dataset"].unique()]
    
    # Set up consistent bar width based on maximum possible classes
    n_models = len(models)
    bar_width = 0.8 / n_models
    
    for col_idx, dataset in enumerate(datasets):
        # Get classes for this specific dataset (using abbreviated names)
        present = df[df['dataset'] == dataset]['class_abbrev'].unique()
        dataset_classes = [c for c in class_order if c in present]
        
        for row_idx, metric in enumerate(metrics):
            ax = axes[row_idx, col_idx]
            
            # Filter data for this dataset and metric
            df_subset = df[(df['dataset'] == dataset) & (df['metric'] == metric)]
            
            # Store bar positions and heights for annotation
            bar_info = {model: {'x': [], 'height': []} for model in models}
            
            # For each model, plot bars
            for i, model in enumerate(models):
                df_model = df_subset[df_subset['model_abbrev'] == model]
                
                # Get values in class order
                values = []
                errors_low = []
                errors_high = []
                
                for cls in dataset_classes:
                    cls_data = df_model[df_model['class_abbrev'] == cls]
                    if len(cls_data) > 0:
                        values.append(cls_data['median'].values[0])
                        errors_low.append(cls_data['median'].values[0] - cls_data['low'].values[0])
                        errors_high.append(cls_data['high'].values[0] - cls_data['median'].values[0])
                    else:
                        values.append(0)
                        errors_low.append(0)
                        errors_high.append(0)
                
                # Calculate bar positions
                n_classes = df_subset.class_abbrev.nunique()
                x_pos = np.arange(n_classes) + i * bar_width - (n_models - 1) * bar_width / 2
                
                # Store for later
                bar_info[model]['x'] = x_pos
                bar_info[model]['height'] = np.array(values) + np.array(errors_high)
                
                # Plot bars
                bars = ax.bar(
                    x_pos,
                    values,
                    bar_width,
                    label=model if row_idx == 0 and col_idx == n_datasets - 1 else "",
                    color=palette[model],
                    alpha=0.8,
                    yerr=[errors_low, errors_high],
                    capsize=3,
                    error_kw={'linewidth': 1.5}
                )
                
                # Annotate bars with values (above error bars)
                for bar, val, err_high in zip(bars, values, errors_high):
                    if val > 0:  # Only annotate non-zero bars
                        # Position text just above the top of the error bar
                        height = val + err_high
                        ax.text(
                            bar.get_x() + bar.get_width() / 2,
                            height + 0.02,
                            f"{val:.2f}",
                            ha="center",
                            va="bottom",
                            fontsize=7,
                            color="black",
                        )
            
            # Add p-value annotations
            pval_subset = pvalues[
                (pvalues['dataset'] == dataset) & 
                (pvalues['metric'] == metric) &
                # (pvalues['p_value'] < p_threshold)
                (pvalues['p_value_bh'] < p_threshold) # use the BH corrected p values
            ]
            
            # Find the maximum bar height across all classes to set consistent baseline
            max_bar_height = 0
            for model in models:
                max_bar_height = max(max_bar_height, np.max(bar_info[model]['height']))
            
            for cls_idx, cls in enumerate(dataset_classes):
                pval_cls = pval_subset[pval_subset['class_abbrev'] == cls]
                
                if len(pval_cls) == 0:
                    continue
                
                # Determine which comparisons to show
                if not show_all_comparisons:
                    # Only show comparisons involving the first model (baseline)
                    baseline_model = models[0]
                    comparisons_to_show = pval_cls[
                        (pval_cls['model1_abbrev'] == baseline_model) | 
                        (pval_cls['model2_abbrev'] == baseline_model)
                    ]
                else:
                    # Show all comparisons
                    comparisons_to_show = pval_cls
                
                # Sort by p-value to show most significant first
                comparisons_to_show = comparisons_to_show.sort_values('p_value_bh')
                
                # Consistent vertical spacing parameters
                y_offset = 0.15  # Start height for brackets above tallest bar
                y_step = 0.12    # Vertical spacing between brackets
                
                for comp_idx, (_, row_pval) in enumerate(comparisons_to_show.iterrows()):
                    model1 = row_pval['model1_abbrev']
                    model2 = row_pval['model2_abbrev']
                    p_val = row_pval['p_value_bh']
                    
                    # Get model indices
                    if model1 not in models or model2 not in models:
                        continue
                    
                    # Get x positions
                    x1 = bar_info[model1]['x'][cls_idx]
                    x2 = bar_info[model2]['x'][cls_idx]
                    
                    # Calculate bracket position using consistent baseline
                    y_bracket = max_bar_height + y_offset + comp_idx * y_step
                    
                    # Format p-value as stars
                    p_text = get_significance_marker(p_val)
                    # Alternative: use actual p-values
                    # p_text = format_pvalue(p_val)
                    
                    # Draw bracket
                    ax.plot([x1, x1, x2, x2], 
                           [y_bracket - 0.01, y_bracket, y_bracket, y_bracket - 0.01],
                           'k-', linewidth=1)
                    
                    # Add p-value text
                    ax.text((x1 + x2) / 2, y_bracket + 0.005,
                           p_text,
                           ha='center', va='bottom', fontsize=7, fontweight='bold')
            
            # Formatting
            ax.set_ylabel(metric_names[metric], fontsize=10)
            
            # Title: dataset name on top row
            if row_idx == 0:
                ax.set_title(f"{dataset}", fontsize=11)
            
            # Use fixed-order class labels
            ax.set_xticks(np.arange(len(dataset_classes)))
            ax.set_xticklabels(dataset_classes, rotation=0, ha='right')
            
            # Adjust y-limit to accommodate brackets
            max_y = 1.2
            if len(pval_subset[pval_subset['class_abbrev'].isin(dataset_classes)]) > 0:
                # Count maximum number of brackets for any class in this panel
                max_brackets = 0
                for cls in dataset_classes:
                    pval_cls = pval_subset[pval_subset['class_abbrev'] == cls]
                    if not show_all_comparisons:
                        baseline_model = models[0]
                        n_brackets = len(pval_cls[
                            (pval_cls['model1_abbrev'] == baseline_model) | 
                            (pval_cls['model2_abbrev'] == baseline_model)
                        ])
                    else:
                        n_brackets = len(pval_cls)
                    max_brackets = max(max_brackets, n_brackets)
                
                max_y = 1.2 + max_brackets * y_step
            ax.set_ylim(0, max_y)
            
            # Set y-ticks to only show values up to 1.0
            yticks = ax.get_yticks()
            yticks_filtered = yticks[yticks <= 1.0]
            ax.set_yticks(yticks_filtered)
            ax.set_yticklabels([f'{y:.1f}' for y in yticks_filtered])
            
            # Only show horizontal grid lines
            ax.grid(True, alpha=0.3, axis='y')
            ax.grid(False, axis='x')
            
            # Remove spines
            sns.despine(ax=ax, left=True, bottom=True, right=True, top=True)
            # Re-add spines only for NACC panels
            if dataset == "NACC":
                for spine in ["left", "bottom", "right", "top"]:
                    ax.spines[spine].set_visible(True)
                    ax.spines[spine].set_linewidth(1.0)
    
    # Add legend
    handles = [
        plt.Rectangle((0, 0), 1, 1, fc=palette[m], alpha=0.8, label=m)
        for m in models
    ]
    fig.legend(handles=handles, title="Model", loc="lower center", ncol=3, 
               bbox_to_anchor=(0.5, -0.053), frameon=True)

    axes[0,0].set_title('NACC (Internal testing)')
    axes[0,1].set_title('All other cohorts (External testing)') 
    
    plt.tight_layout(rect=[0, 0.02, 1, 1])
    
    # Save figure
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.join(output_dir, "classwise_precision_recall_with_pvalues.pdf")
    plt.savefig(filename, bbox_inches="tight", dpi=300)
    print(f"Saved {filename}")
    plt.close()

In [22]:
# Create the plot with p-value annotations
plot_classwise_with_pvalues(
    all_metrics=all_metrics,
    pairwise_pvalues=pairwise_pvalues,
    model_map=model_map,
    class_map=class_map,
    output_dir="../figures/figure2_classwise",
    show_all_comparisons=True,  # Only show comparisons with baseline (Q3B)
    p_threshold=1  # Only show significant results
)

Saved ../figures/figure2_classwise/classwise_precision_recall_with_pvalues.pdf
